<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/19-sequences-and-transformers/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 19: Exercises

Twelve exercises: four Easy, five Medium and three Hard. The Easy ones are the small tools every language model needs: encoding text, the causal mask, cutting training examples, and sampling with temperature and top-k. The Medium ones build the parts of the lesson yourself: the bigram model, the merge step of BPE, attention, the position encodings of the original transformer, and the generation loop. The Hard ones put it together: training a BPE tokeniser, multi-head attention, and a full transformer block, the last two checked by copying the weights of PyTorch's own versions into yours.

Everything runs in a few seconds on a CPU, and nothing is downloaded.

**Run the setup cell first.**

Rules:

- Python, NumPy and PyTorch. Don't call the PyTorch version of what you're writing (such as `F.scaled_dot_product_attention` in Exercise 7); the checks use it to test your answer.
- Solutions are in `03_solutions.ipynb`.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import collections
import math

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, torch.Tensor) or isinstance(b, torch.Tensor):
        if not (isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor)):
            return False
        a, b = a.detach().cpu().numpy(), b.detach().cpu().numpy()
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol, equal_nan=True))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k], tol) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, torch.Tensor):
        body = np.array2string(x.detach().cpu().numpy(), precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"tensor(shape={tuple(x.shape)}, dtype={str(x.dtype).replace('torch.', '')}, {' '.join(body.split())})"
    elif isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, (list, tuple)) and any(isinstance(i, (np.ndarray, torch.Tensor)) for i in x):
        brackets = "()" if isinstance(x, tuple) else "[]"
        return brackets[0] + ",\n           ".join(_short(i, limit) for i in x) + brackets[1]
    else:
        text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def check_encode_chars(f):
    vocab = list("\n !',.:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz")
    line = "To be, or not to be: that is the question."
    _scenarios("encode_chars", [
        ("the example", lambda: f("abba", ["a", "b"]), [0, 1, 1, 0], None),
        ("a line of Shakespeare", lambda: f(line, vocab), [vocab.index(c) for c in line], "a character's number is its position in vocab"),
        ("an unsorted vocabulary", lambda: f("cab", ["c", "a", "b"]), [0, 1, 2], None),
        ("the empty string", lambda: f("", vocab), [], None),
        ("returns a list of ints", lambda: [type(f("ab", ["a", "b"])).__name__] + sorted({type(i).__name__ for i in f("ab", ["a", "b"])}),
         ["list", "int"], None),
        ("an unknown character raises", lambda: f("café", vocab), Raises(ValueError), None),
    ])


def check_causal_mask(f):
    _scenarios("causal_mask", [
        ("the example", lambda: f(3), torch.tensor([[False, True, True], [False, False, True], [False, False, False]]), None),
        ("size 8", lambda: f(8), torch.triu(torch.ones(8, 8, dtype=torch.bool), diagonal=1), "True above the diagonal, False on and below it"),
        ("dtype", lambda: str(f(4).dtype), "torch.bool", None),
        ("a single position can see itself", lambda: f(1), torch.tensor([[False]]), None),
    ])


def check_next_token_pairs(f):
    data = torch.arange(100, 200)
    starts = [0, 17, 63, 89]
    _scenarios("next_token_pairs", [
        ("the example", lambda: f(torch.arange(10), 3, [0, 5]), (torch.tensor([[0, 1, 2], [5, 6, 7]]), torch.tensor([[1, 2, 3], [6, 7, 8]])), None),
        ("four windows", lambda: f(data, 10, starts),
         (torch.stack([data[s:s + 10] for s in starts]), torch.stack([data[s + 1:s + 11] for s in starts])), "y is x shifted by one"),
        ("the last valid start", lambda: f(data, 10, [89])[1][0, -1].item(), 199, None),
        ("a start that's too late raises", lambda: f(data, 10, [90]), Raises(ValueError), "the target needs block_size + 1 tokens"),
        ("a negative start raises", lambda: f(data, 10, [-5]), Raises(ValueError), None),
    ])


def check_temperature_top_k(f):
    g = torch.Generator().manual_seed(0)
    logits = torch.randn(4, 10, generator=g, dtype=torch.float64) * 2

    def reference(x, temperature=1.0, k=None):
        x = x / temperature
        if k is not None:
            kth = x.topk(k, dim=-1).values[..., -1:]
            x = x.masked_fill(x < kth, float("-inf"))
        return torch.softmax(x, dim=-1)

    tied = torch.tensor([3.0, 1.0, 1.0, 0.0], dtype=torch.float64)
    _scenarios("temperature_top_k", [
        ("the example", lambda: f(torch.tensor([2.0, 1.0, 0.0]), k=2), torch.tensor([0.7310586, 0.2689414, 0.0]), None),
        ("temperature 1 is a plain softmax", lambda: f(logits), torch.softmax(logits, dim=-1), None),
        ("temperature 0.5 sharpens", lambda: f(logits, temperature=0.5), reference(logits, 0.5), "divide the logits by the temperature"),
        ("temperature 2 with top-3, a batch of rows", lambda: f(logits, temperature=2.0, k=3), reference(logits, 2.0, 3),
         "keep the k largest in each row, then renormalise"),
        ("ties with the k-th value are kept", lambda: f(tied, k=2), reference(tied, k=2), None),
        ("temperature 0 raises", lambda: f(logits, temperature=0), Raises(ValueError), None),
    ])


def check_bigram_probabilities(f):
    g = torch.Generator().manual_seed(1)
    tokens = torch.randint(0, 6, (500,), generator=g)

    def reference(tok, v, s):
        counts = np.zeros((v, v))
        for a, b in zip(tok[:-1].tolist(), tok[1:].tolist()):
            counts[a, b] += 1
        counts += s
        return torch.tensor(counts / counts.sum(axis=1, keepdims=True), dtype=torch.float32)

    _scenarios("bigram_probabilities", [
        ("the example", lambda: f([0, 1, 0, 1, 1], 2)[0], torch.tensor([0.25, 0.75]), None),
        ("500 random tokens, repeated pairs counted every time", lambda: f(tokens, 6), reference(tokens, 6, 1.0),
         "repeated pairs must each add 1: use index_put_ with accumulate=True, or a loop"),
        ("smoothing 0.1", lambda: f(tokens, 6, smoothing=0.1), reference(tokens, 6, 0.1), None),
        ("a token with no data gets a uniform row", lambda: f([0, 1, 0], 3)[2], torch.full((3,), 1 / 3), None),
        ("rows add up to 1", lambda: f(tokens, 8).sum(dim=1), torch.ones(8), None),
        ("smoothing 0 raises", lambda: f(tokens, 6, smoothing=0), Raises(ValueError), None),
    ])


def check_merge_pair(f):
    original = ["t", "h", "e", " ", "t", "h", "e", "n"]

    def unchanged():
        f(original, ("t", "h"), "th")
        return original

    _scenarios("merge_pair", [
        ("the first example", lambda: f(["l", "o", "w", "l", "o"], ("l", "o"), "lo"), ["lo", "w", "lo"], None),
        ("overlapping occurrences", lambda: f(["a", "a", "a"], ("a", "a"), "aa"), ["aa", "a"], "after a merge, continue after both tokens"),
        ("four in a row", lambda: f(["a", "a", "a", "a"], ("a", "a"), "aa"), ["aa", "aa"], None),
        ("numbers", lambda: f([1, 2, 3, 1, 2], (1, 2), 99), [99, 3, 99], None),
        ("no occurrence", lambda: f(["x", "y"], ("y", "x"), "yx"), ["x", "y"], None),
        ("only the first half matches", lambda: f(["a", "c", "a", "b"], ("a", "b"), "ab"), ["a", "c", "ab"], "both tokens of the pair must match"),
        ("the pair at the very end", lambda: f(["a", "b", "c"], ("b", "c"), "bc"), ["a", "bc"], None),
        ("the input list is not changed", unchanged, ["t", "h", "e", " ", "t", "h", "e", "n"], None),
    ])


def check_scaled_dot_product_attention(f):
    g = torch.Generator().manual_seed(2)
    q = torch.randn(2, 3, 7, 8, generator=g, dtype=torch.float64)
    k = torch.randn(2, 3, 7, 8, generator=g, dtype=torch.float64)
    v = torch.randn(2, 3, 7, 5, generator=g, dtype=torch.float64)
    k_long = torch.randn(2, 3, 9, 8, generator=g, dtype=torch.float64)
    v_long = torch.randn(2, 3, 9, 5, generator=g, dtype=torch.float64)

    def weights_of(causal):
        return f(q, k, v, causal=causal)[1]

    _scenarios("scaled_dot_product_attention", [
        ("causal output", lambda: f(q, k, v)[0], F.scaled_dot_product_attention(q, k, v, is_causal=True),
         "scale by sqrt of the query size, and mask the future"),
        ("weights are zero above the diagonal", lambda: weights_of(True).triu(diagonal=1).abs().max().item(), 0.0, None),
        ("weights add up to 1", lambda: weights_of(True).sum(dim=-1), torch.ones(2, 3, 7, dtype=torch.float64), None),
        ("without the mask", lambda: f(q, k, v, causal=False)[0], F.scaled_dot_product_attention(q, k, v), None),
        ("cross-attention: 7 queries, 9 keys", lambda: f(q, k_long, v_long, causal=False)[0],
         F.scaled_dot_product_attention(q, k_long, v_long), "use transpose(-2, -1), not .T"),
        ("a single sequence without batch dimensions", lambda: f(q[0, 0], k[0, 0], v[0, 0])[0],
         F.scaled_dot_product_attention(q[0, 0][None], k[0, 0][None], v[0, 0][None], is_causal=True)[0], None),
    ])


def check_sinusoidal_positions(f):
    def reference(n, d):
        pe = np.zeros((n, d))
        for pos in range(n):
            for i in range(d // 2):
                angle = pos / 10000 ** (2 * i / d)
                pe[pos, 2 * i] = math.sin(angle)
                pe[pos, 2 * i + 1] = math.cos(angle)
        return torch.tensor(pe, dtype=torch.float32)

    _scenarios("sinusoidal_positions", [
        ("row 0", lambda: f(4, 6)[0], torch.tensor([0.0, 1, 0, 1, 0, 1]), None),
        ("a small table", lambda: f(10, 8), reference(10, 8), "the exponent is 2 * i / d_model, and sines and cosines alternate"),
        ("a larger table", lambda: f(64, 32), reference(64, 32), None),
        ("dtype and shape", lambda: (str(f(5, 4).dtype), tuple(f(5, 4).shape)), ("torch.float32", (5, 4)), None),
        ("an odd size raises", lambda: f(5, 7), Raises(ValueError), None),
    ])


class _NextPlusOne(nn.Module):
    """Predicts (token + 1) mod 10 at every position; refuses inputs longer than 6."""

    def __init__(self):
        super().__init__()
        self.dummy = nn.Linear(1, 1)

    def forward(self, idx):
        if idx.shape[1] > 6:
            raise RuntimeError("input longer than the block size")
        return F.one_hot((idx + 1) % 10, 10).float() * 5


class _OwnPlusThree(nn.Module):
    """At every position, predicts that position's own token plus 3."""

    def forward(self, idx):
        return F.one_hot((idx + 3) % 10, 10).float()


class _ModeSpy(nn.Module):
    def __init__(self):
        super().__init__()
        self.drop = nn.Dropout(0.5)
        self.modes, self.grads = [], []

    def forward(self, idx):
        self.modes.append(self.training)
        self.grads.append(torch.is_grad_enabled())
        return F.one_hot(idx % 10, 10).float()


def check_generate_greedy(f):
    spy = _ModeSpy()

    def spied():
        f(spy, torch.tensor([[1, 2]]), 3, 6)
        return (all(not m for m in spy.modes), any(spy.grads))

    _scenarios("generate_greedy", [
        ("counts upwards", lambda: f(_NextPlusOne(), torch.tensor([[3]]), 5, 6), torch.tensor([[3, 4, 5, 6, 7, 8]]), None),
        ("crops the context to block_size", lambda: f(_NextPlusOne(), torch.tensor([[0, 1]]), 10, 6)[0, -1].item(), 1,
         "give the model only the last block_size tokens"),
        ("uses the prediction at the last position", lambda: f(_OwnPlusThree(), torch.tensor([[1, 2, 3]]), 1, 6),
         torch.tensor([[1, 2, 3, 6]]), "logits[:, -1, :] is the prediction for the next token"),
        ("a batch of two", lambda: f(_NextPlusOne(), torch.tensor([[1], [7]]), 3, 6), torch.tensor([[1, 2, 3, 4], [7, 8, 9, 0]]), None),
        ("evaluation mode, no gradients", spied, (True, False), "call model.eval() and use torch.no_grad()"),
    ])


def _reference_bpe(word_counts, n_merges):
    split = {w: list(w) for w in word_counts}
    merges = []
    for _ in range(n_merges):
        pairs = collections.Counter()
        for w, n in word_counts.items():
            p = split[w]
            for pair in zip(p, p[1:]):
                pairs[pair] += n
        if not pairs:
            break
        best = max(pairs, key=lambda q: (pairs[q], q))
        merges.append(best)
        for w in split:
            p, out, i = split[w], [], 0
            while i < len(p):
                if i + 1 < len(p) and (p[i], p[i + 1]) == best:
                    out.append(p[i] + p[i + 1])
                    i += 2
                else:
                    out.append(p[i])
                    i += 1
            split[w] = out
    return merges


def check_bpe_train(f):
    classic = {"low": 5, "lower": 2, "newest": 6, "widest": 3}
    sample = ("the quick brown fox jumps over the lazy dog and the dog barks back at the fox "
              "while the other foxes and dogs watch the show from the hill over there ") * 3
    sample_counts = dict(collections.Counter(" " + w for w in sample.split()))
    _scenarios("bpe_train", [
        ("the first two merges", lambda: f(classic, 2), [("s", "t"), ("e", "st")], "ties go to the larger pair: max by (count, pair)"),
        ("ten merges", lambda: f(classic, 10), _reference_bpe(classic, 10), None),
        ("counts weight the pairs", lambda: f({"ab": 1, "cd": 5}, 1), [("c", "d")], "add the word's count, not 1"),
        ("stops when every word is one token", lambda: f({"ab": 3, "c": 1}, 5), [("a", "b")], None),
        ("overlaps inside a word", lambda: f({"aaaa": 1, "aaa": 2}, 3), _reference_bpe({"aaaa": 1, "aaa": 2}, 3), None),
        ("a larger sample", lambda: f(sample_counts, 30), _reference_bpe(sample_counts, 30), None),
    ])


def _causal_bool(t):
    return torch.triu(torch.ones(t, t, dtype=torch.bool), diagonal=1)


def _torch_mha(d, h):
    torch.manual_seed(0)
    mha = nn.MultiheadAttention(d, h, batch_first=True).double()
    with torch.no_grad():
        mha.in_proj_bias.normal_(0, 0.2)
        mha.out_proj.bias.normal_(0, 0.2)
    return mha


def check_MultiHeadAttention(cls):
    x = torch.randn(3, 7, 16, generator=torch.Generator().manual_seed(3), dtype=torch.float64)

    def yours(d, h):
        reference = _torch_mha(d, h)
        mine = cls(d, h).double()
        with torch.no_grad():
            mine.qkv.weight.copy_(reference.in_proj_weight)
            mine.qkv.bias.copy_(reference.in_proj_bias)
            mine.proj.weight.copy_(reference.out_proj.weight)
            mine.proj.bias.copy_(reference.out_proj.bias)
        return mine(x[..., :d])

    def theirs(d, h):
        reference = _torch_mha(d, h)
        out, _ = reference(x[..., :d], x[..., :d], x[..., :d], attn_mask=_causal_bool(7), need_weights=False)
        return out.detach()

    def future_leak():
        mine = cls(16, 4).double()
        a = x.clone()
        b = x.clone()
        b[:, 5:] = 0
        with torch.no_grad():
            return (mine(a)[:, :5] - mine(b)[:, :5]).abs().max().item()

    _scenarios("MultiHeadAttention", [
        ("output shape", lambda: tuple(cls(16, 4).double()(x).shape), (3, 7, 16), None),
        ("parameters", lambda: sum(p.numel() for p in cls(16, 4).parameters()), 4 * 16 * 16 + 4 * 16, None),
        ("the future can't affect the past", future_leak, 0.0, "apply the causal mask"),
        ("matches nn.MultiheadAttention, 4 heads", lambda: yours(16, 4), theirs(16, 4),
         "view as (b, t, heads, head_size), then transpose(1, 2)"),
        ("matches nn.MultiheadAttention, 2 heads", lambda: yours(16, 2), theirs(16, 2), None),
        ("matches nn.MultiheadAttention, 1 head", lambda: yours(8, 1), theirs(8, 1), None),
        ("d_model not divisible by n_heads raises", lambda: cls(10, 4), Raises(ValueError), None),
    ])


def _torch_block(d, h):
    torch.manual_seed(0)
    layer = nn.TransformerEncoderLayer(d, h, dim_feedforward=4 * d, dropout=0.0, activation="gelu",
                                       batch_first=True, norm_first=True).double()
    with torch.no_grad():
        for p in layer.parameters():
            if p.dim() == 1:
                p.normal_(0, 0.2)
        layer.norm1.weight.add_(1.0)
        layer.norm2.weight.add_(1.0)
    return layer


def check_TransformerBlock(cls):
    x = torch.randn(2, 6, 16, generator=torch.Generator().manual_seed(4), dtype=torch.float64)

    def yours(d, h):
        ref = _torch_block(d, h)
        mine = cls(d, h).double()
        with torch.no_grad():
            mine.ln1.weight.copy_(ref.norm1.weight)
            mine.ln1.bias.copy_(ref.norm1.bias)
            mine.ln2.weight.copy_(ref.norm2.weight)
            mine.ln2.bias.copy_(ref.norm2.bias)
            mine.attn.qkv.weight.copy_(ref.self_attn.in_proj_weight)
            mine.attn.qkv.bias.copy_(ref.self_attn.in_proj_bias)
            mine.attn.proj.weight.copy_(ref.self_attn.out_proj.weight)
            mine.attn.proj.bias.copy_(ref.self_attn.out_proj.bias)
            mine.mlp[0].weight.copy_(ref.linear1.weight)
            mine.mlp[0].bias.copy_(ref.linear1.bias)
            mine.mlp[2].weight.copy_(ref.linear2.weight)
            mine.mlp[2].bias.copy_(ref.linear2.bias)
        return mine(x[..., :d])

    def theirs(d, h):
        ref = _torch_block(d, h)
        ref.train()                                         # avoids PyTorch's separate fast path; no dropout anyway
        return ref(x[..., :d], src_mask=_causal_bool(6)).detach()

    _scenarios("TransformerBlock", [
        ("output shape", lambda: tuple(cls(16, 4).double()(x).shape), (2, 6, 16), None),
        ("parameters", lambda: sum(p.numel() for p in cls(16, 4).parameters()),
         sum(p.numel() for p in _torch_block(16, 4).parameters()), "the MLP is 4 times wider than d_model"),
        ("layer types", lambda: [type(m).__name__ for m in cls(16, 4).mlp], ["Linear", "GELU", "Linear"], None),
        ("matches nn.TransformerEncoderLayer, 4 heads", lambda: yours(16, 4), theirs(16, 4),
         "pre-norm: x = x + attn(ln1(x)), then x = x + mlp(ln2(x))"),
        ("matches nn.TransformerEncoderLayer, 2 heads", lambda: yours(8, 2), theirs(8, 2), None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `encode_chars`

Write `encode_chars(text, vocab)` that turns a string into a list of token numbers, one per character.

- `vocab` is a list of characters; a character's number is its position in the list.
- Return a plain Python list of `int`.
- Raise a `ValueError` if the text contains a character that isn't in `vocab`, and say which character in the message.

Example: `encode_chars("abba", ["a", "b"])` is `[0, 1, 1, 0]`.

In [2]:
def encode_chars(text: str, vocab) -> list:
    # your code here
    pass

In [3]:
check_encode_chars(encode_chars)

FAIL  encode_chars: the example
      expected: [0, 1, 1, 0]
      got:      None


### Exercise 2: `causal_mask`

Write `causal_mask(t)` that returns the mask used in a GPT's attention for a sequence of length `t`.

- Return a `torch.bool` tensor of shape `(t, t)`.
- Entry `[i, j]` is `True` when position `i` must **not** look at position `j`, that is, when `j` is in the future (`j > i`).

Example: `causal_mask(3)` is
```
[[False,  True,  True],
 [False, False,  True],
 [False, False, False]]
```

In [4]:
def causal_mask(t: int) -> torch.Tensor:
    # your code here
    pass

In [5]:
check_causal_mask(causal_mask)

FAIL  causal_mask: the example
      expected: tensor(shape=(3, 3), dtype=bool, [[False, True, True], [False, False, True], [False, Fa...
      got:      None


### Exercise 3: `next_token_pairs`

Write `next_token_pairs(data, block_size, starts)` that cuts training examples for a language model out of a long sequence of token numbers.

- `data` is a 1D tensor of token numbers; `starts` is a list of starting positions.
- For each start `s`, the input is `data[s : s + block_size]` and the target is the same window shifted one place to the right, `data[s + 1 : s + block_size + 1]`.
- Return `(x, y)`, two tensors of shape `(len(starts), block_size)`.
- Raise a `ValueError` if a start is negative or leaves too little data for its target.

Example: with `data = torch.arange(10)`, `block_size=3` and `starts=[0, 5]`, `x` is `[[0, 1, 2], [5, 6, 7]]` and `y` is `[[1, 2, 3], [6, 7, 8]]`.

In [6]:
def next_token_pairs(data: torch.Tensor, block_size: int, starts):
    # your code here
    pass

In [7]:
check_next_token_pairs(next_token_pairs)

FAIL  next_token_pairs: the example
      expected: (tensor(shape=(2, 3), dtype=int64, [[0, 1, 2], [5, 6, 7]]),
           tensor(shape=(2, 3), dtype=int64, [[1, 2, 3], [6, 7, 8]]))
      got:      None


### Exercise 4: `temperature_top_k`

Write `temperature_top_k(logits, temperature=1.0, k=None)` that turns a model's logits into the probabilities used for sampling.

- Divide the logits by `temperature`, then apply softmax over the **last** dimension.
- If `k` is given, keep only the `k` largest logits in each row (and any that tie with the `k`-th largest): set the others' probabilities to 0, so the kept ones add up to 1.
- Raise a `ValueError` if `temperature` isn't positive.
- Work for a single vector of logits or a batch of rows.

Example: `temperature_top_k(torch.tensor([2.0, 1.0, 0.0]), k=2)` is about `[0.731, 0.269, 0.0]`.

In [8]:
def temperature_top_k(logits: torch.Tensor, temperature: float = 1.0, k=None) -> torch.Tensor:
    # your code here
    pass

In [9]:
check_temperature_top_k(temperature_top_k)

FAIL  temperature_top_k: the example
      expected: tensor(shape=(3,), dtype=float32, [0.7311, 0.2689, 0. ])
      got:      None


## Medium

### Exercise 5: `bigram_probabilities`

Write `bigram_probabilities(tokens, vocab_size, smoothing=1.0)` that builds the bigram model from Section 3 of the lesson.

- `tokens` is a list or 1D tensor of token numbers.
- Count how often each token `j` follows each token `i`, add `smoothing` to every count, and divide each row by its total.
- Return a `float` tensor `P` of shape `(vocab_size, vocab_size)` where `P[i, j]` is the probability that `j` comes next after `i`. Every row adds up to 1.
- Raise a `ValueError` if `smoothing` isn't positive.

Example: for `tokens = [0, 1, 0, 1, 1]` with `vocab_size=2` and `smoothing=1`, token 0 is followed by 1 twice and by 0 never, so `P[0]` is `[1/4, 3/4]`.

In [10]:
def bigram_probabilities(tokens, vocab_size: int, smoothing: float = 1.0) -> torch.Tensor:
    # your code here
    pass

In [11]:
check_bigram_probabilities(bigram_probabilities)

FAIL  bigram_probabilities: the example
      raised TypeError: 'NoneType' object is not subscriptable
      expected: tensor(shape=(2,), dtype=float32, [0.25, 0.75])


### Exercise 6: `merge_pair`

Write `merge_pair(tokens, pair, new_token)`, the basic step of byte-pair encoding.

- `tokens` is a list (of strings, numbers, or anything comparable).
- Scan from left to right, replacing each occurrence of the two neighbouring tokens `pair[0], pair[1]` with `new_token`.
- Occurrences don't overlap: once two tokens are merged, the scan continues after them.
- Return a new list; don't change `tokens`.

Examples: `merge_pair(["l", "o", "w", "l", "o"], ("l", "o"), "lo")` is `["lo", "w", "lo"]`, and `merge_pair(["a", "a", "a"], ("a", "a"), "aa")` is `["aa", "a"]`.

In [12]:
def merge_pair(tokens, pair, new_token) -> list:
    # your code here
    pass

In [13]:
check_merge_pair(merge_pair)

FAIL  merge_pair: the first example
      expected: ['lo', 'w', 'lo']
      got:      None


### Exercise 7: `scaled_dot_product_attention`

Write `scaled_dot_product_attention(q, k, v, causal=True)` that computes attention as in Section 7 of the lesson, and returns both the output and the attention weights.

- `q` has shape `(..., T_q, d)`, `k` has shape `(..., T_k, d)` and `v` has shape `(..., T_k, d_v)`, where `...` is any number of batch or head dimensions.
- Scores are `q @ k^T / sqrt(d)`. If `causal` is true, block every future position (`T_q` and `T_k` are then equal). Softmax over the last dimension gives the weights.
- Return `(weights @ v, weights)`.
- Don't use `F.scaled_dot_product_attention`; the check compares your output with it.

In [14]:
def scaled_dot_product_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, causal: bool = True):
    # your code here
    pass

In [15]:
check_scaled_dot_product_attention(scaled_dot_product_attention)

FAIL  scaled_dot_product_attention: causal output
      raised TypeError: 'NoneType' object is not subscriptable
      expected: tensor(shape=(2, 3, 7, 5), dtype=float64, [[[[ 0.9254, -1.1614, 1.8581, -0.2948, -2.064...
      hint: scale by sqrt of the query size, and mask the future


### Exercise 8: `sinusoidal_positions`

Write `sinusoidal_positions(n_positions, d_model)` that builds the fixed position encodings from the original transformer paper (Vaswani and colleagues, 2017), an alternative to the learnt position table used in the lesson.

Return a `float32` tensor `PE` of shape `(n_positions, d_model)` where, for each position `pos` and each `i` from `0` to `d_model / 2 - 1`:

- `PE[pos, 2 * i] = sin(pos / 10000 ** (2 * i / d_model))`
- `PE[pos, 2 * i + 1] = cos(pos / 10000 ** (2 * i / d_model))`

Raise a `ValueError` if `d_model` is odd.

Example: row 0 is `[0, 1, 0, 1, ...]`, because `sin(0) = 0` and `cos(0) = 1`.

In [16]:
def sinusoidal_positions(n_positions: int, d_model: int) -> torch.Tensor:
    # your code here
    pass

In [17]:
check_sinusoidal_positions(sinusoidal_positions)

FAIL  sinusoidal_positions: row 0
      raised TypeError: 'NoneType' object is not subscriptable
      expected: tensor(shape=(6,), dtype=float32, [0., 1., 0., 1., 0., 1.])


### Exercise 9: `generate_greedy`

Write `generate_greedy(model, idx, n_new, block_size)` that extends a sequence one token at a time, always choosing the most likely next token.

- `model` takes a tensor of token numbers of shape `(batch, t)` and returns logits of shape `(batch, t, vocab_size)`. It only accepts `t <= block_size`.
- `idx` is the starting sequence, shape `(batch, t0)`.
- At each step, give the model the **last** `block_size` tokens (or all of them, if there are fewer), take the logits at the last position, pick the largest with `argmax`, and append it.
- Return the full sequence, shape `(batch, t0 + n_new)`.
- Put the model in evaluation mode and don't track gradients.

In [18]:
def generate_greedy(model: nn.Module, idx: torch.Tensor, n_new: int, block_size: int) -> torch.Tensor:
    # your code here
    pass

In [19]:
check_generate_greedy(generate_greedy)

FAIL  generate_greedy: counts upwards
      expected: tensor(shape=(1, 6), dtype=int64, [[3, 4, 5, 6, 7, 8]])
      got:      None


## Hard

### Exercise 10: `bpe_train`

Write `bpe_train(word_counts, n_merges)` that learns byte-pair encoding merges.

- `word_counts` is a dictionary from words (strings) to how many times they appear in the text. Each word starts as a sequence of single characters, and merges never cross from one word into another.
- Repeat `n_merges` times:
  1. Count every pair of neighbouring tokens across all words, **weighted** by the word's count.
  2. Choose the pair with the highest count. Break ties by choosing the pair that is largest in Python's ordering of tuples, so that `max(pairs, key=lambda p: (count[p], p))` gives the answer.
  3. Merge that pair everywhere (left to right, without overlaps, as in Exercise 6), joining the two strings into one new token.
- Stop early if no pairs are left (every word is a single token).
- Return the list of merged pairs, in order, as tuples of two strings.

Example: for `{"low": 5, "lower": 2, "newest": 6, "widest": 3}`, the pairs `("e", "s")` and `("s", "t")` both appear 9 times (in "newest" and "widest"); the tie goes to the larger pair, so the first merge is `("s", "t")`. The second is `("e", "st")`.

In [20]:
def bpe_train(word_counts: dict, n_merges: int) -> list:
    # your code here
    pass

In [21]:
check_bpe_train(bpe_train)

FAIL  bpe_train: the first two merges
      expected: [('s', 't'), ('e', 'st')]
      got:      None
      hint: ties go to the larger pair: max by (count, pair)


### Exercise 11: `MultiHeadAttention`

Write a class `MultiHeadAttention(d_model, n_heads)`, causal multi-head self-attention as an `nn.Module`, with exactly these layers:

- `self.qkv = nn.Linear(d_model, 3 * d_model)`: computes the queries, keys and values for all heads at once, in that order (the first `d_model` outputs are the queries).
- `self.proj = nn.Linear(d_model, d_model)`: mixes the heads' outputs.

`forward(x)` takes `x` of shape `(batch, t, d_model)` and:

1. splits `self.qkv(x)` into queries, keys and values, each `(batch, t, d_model)`;
2. splits each into `n_heads` heads of size `d_model // n_heads`, giving `(batch, n_heads, t, head_size)`;
3. applies causal scaled dot-product attention in every head;
4. joins the heads back into `(batch, t, d_model)` and applies `self.proj`.

Raise a `ValueError` in `__init__` if `d_model` isn't divisible by `n_heads`. You may use your function from Exercise 7 or `F.scaled_dot_product_attention`.

The check copies the weights of PyTorch's `nn.MultiheadAttention` into your module and compares the outputs.

In [22]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        # your code here

    def forward(self, x):
        # your code here
        pass

In [23]:
check_MultiHeadAttention(MultiHeadAttention)

FAIL  MultiHeadAttention: output shape


      raised AttributeError: 'NoneType' object has no attribute 'shape'
      expected: (3, 7, 16)


### Exercise 12: `TransformerBlock`

Write a class `TransformerBlock(d_model, n_heads)`, the pre-norm GPT block from Section 9 of the lesson, using your `MultiHeadAttention` from Exercise 11. Use exactly these attribute names:

- `self.ln1 = nn.LayerNorm(d_model)`
- `self.attn = MultiHeadAttention(d_model, n_heads)`
- `self.ln2 = nn.LayerNorm(d_model)`
- `self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))`

`forward(x)` applies `x = x + attn(ln1(x))`, then `x = x + mlp(ln2(x))`, and returns `x`.

The check copies the weights of PyTorch's own `nn.TransformerEncoderLayer` (set up as a pre-norm, causal, GELU block with no dropout) into yours and compares the outputs. Exercise 11 has to pass first.

In [24]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        # your code here

    def forward(self, x):
        # your code here
        pass

In [25]:
check_TransformerBlock(TransformerBlock)

FAIL  TransformerBlock: output shape
      raised AttributeError: 'NoneType' object has no attribute 'shape'
      expected: (2, 6, 16)


## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then open the mini project in `04_mini_project.ipynb`, where a transformer learns to add.